# Smart Grievance System — Vision Models (Colab)

This notebook trains two small **from-scratch** computer-vision models and exports them to **ONNX** for integration into the Flask app.

## Model 1 — Issue classifier

Classes:

- `pothole`
- `garbage`
- `other` (normal / clean road)
- `water_leak` *(optional)*

## Model 2 — AI-generated image detector

Classes:

- `real`
- `fake`

## Outputs

```text
issue_classifier.onnx
issue_classifier_labels.json
issue_classifier_preprocess.json
issue_classifier_history.json

ai_image_detector.onnx
ai_image_detector_labels.json
ai_image_detector_preprocess.json
ai_image_detector_history.json
```

## Before you start

1. `Runtime -> Change runtime type -> T4 GPU`
2. Create a Kaggle API token: `Kaggle -> Account -> Create New API Token`
3. Run the cells in order
4. Do **not** skip the visual dataset check cell

## Fast path

To finish quickly:

- Keep `USE_GARBAGE_V2 = True`
- Leave `WATER_LEAK_DIR = ''`

That trains three issue classes: `pothole`, `garbage`, `other`.

## Limitations

These are demo-grade models, not production models. The AI-image detector is trained on CIFAKE (`32x32` images) and will not generalise perfectly to phone photos. Never auto-reject a complaint based on the authenticity score.


## Step 1 — Install dependencies

In [ ]:
!pip -q install kaggle onnx onnxruntime onnxscript scikit-learn pandas pillow matplotlib

## Step 2 — Create folders and set random seed

In [ ]:
from pathlib import Path
import os, json, random, shutil
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

BASE = Path('/content/grievance_cv')
RAW = BASE / 'raw'
DATASET = BASE / 'dataset'
OUTPUT = BASE / 'outputs'

for p in (RAW, DATASET, OUTPUT):
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('Base folder:', BASE)
print('Raw folder :', RAW)
print('Dataset    :', DATASET)
print('Outputs    :', OUTPUT)


## Step 3 — Upload your Kaggle API token

Download `kaggle.json` from Kaggle, then run the next cell and upload it.

In [ ]:
from google.colab import files

uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))


In [ ]:
import os
import shutil

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

json_files = [name for name in uploaded if name.lower().endswith('.json')]
if not json_files:
    raise RuntimeError('No JSON file uploaded. Upload kaggle.json and re-run this cell.')

source = json_files[0]
target = os.path.join(kaggle_dir, 'kaggle.json')
shutil.copy(source, target)
os.chmod(target, 0o600)

print('Kaggle token installed at:', target)


## Step 4 — Download datasets

Datasets used:

- Pothole / plain road
- Clean / littered road
- Garbage V2 (extra garbage images)
- CIFAKE (real vs AI-generated)

If one download fails, the notebook continues so you can inspect the problem.

In [ ]:
!kaggle datasets download -d virenbr11/pothole-and-plain-rode-images -p /content/grievance_cv/raw/pothole --unzip || echo "WARNING: pothole dataset download failed"

!kaggle datasets download -d faizalkarim/cleandirty-road-classification -p /content/grievance_cv/raw/litter --unzip || echo "WARNING: litter dataset download failed"

!kaggle datasets download -d sumn2u/garbage-classification-v2 -p /content/grievance_cv/raw/garbage --unzip || echo "WARNING: garbage-v2 dataset download failed"

!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p /content/grievance_cv/raw/cifake --unzip || echo "WARNING: CIFAKE dataset download failed"


## Step 5 — Optional: add water-leak images

Skip this cell if you want the fast path.

1. Open one of the Roboflow water-leak datasets:
   - `https://universe.roboflow.com/water-leak-hwh4x/water-leakage-ayy4s`
   - `https://universe.roboflow.com/roboflow1-rz9go/water-leakage-aiufw`
2. Create a free Roboflow account.
3. Download the dataset as **Folder Structure** or **YOLOv8**.
4. Upload the zip to Colab. Rename it to `water-leak.zip` or edit the filename in the cell below.
5. Uncomment and run the cell below.
6. Set `WATER_LEAK_DIR = '/content/grievance_cv/raw/water'` in the dataset-building cell.

Only put water-leak images in that folder. If the export contains mixed classes, organise the leak images into their own folder first.

In [ ]:
# OPTIONAL: uncomment and run only if you downloaded a Roboflow zip.

# from google.colab import files
# uploaded_water = files.upload()
#
# !mkdir -p /content/grievance_cv/raw/water
# !unzip -q water-leak.zip -d /content/grievance_cv/raw/water
#
# print('Water-leak files extracted.')


## Step 6 — Inspect the downloaded folder structure

Run this cell if a later step cannot find images. It prints the first entries from each dataset.

In [ ]:
for root in [RAW / 'pothole', RAW / 'litter', RAW / 'garbage', RAW / 'cifake']:
    print('\n' + '=' * 70)
    print(root)

    if not root.exists():
        print('MISSING')
        continue

    paths = list(root.rglob('*'))
    print('Entries:', len(paths))

    for p in paths[:40]:
        print(' ', p.relative_to(root))


## Step 7 — Build the issue-classification dataset

This cell:

- Finds pothole and plain-road images
- Parses the clean/littered metadata
- Adds extra garbage images from Garbage V2
- Optionally adds water-leak images
- Splits into `train`, `val`, `test`
- Copies images into class folders

If `clean` and `dirty` are reversed, fix the mapping in `map_litter_label` and rerun this cell.

In [ ]:
# ============================================================
# Configuration
# ============================================================

USE_GARBAGE_V2 = True
MAX_PER_CLASS = 2000
MIN_IMAGES_PER_CLASS = 20

# Set this to your Roboflow folder if you downloaded water-leak images.
# Example: '/content/grievance_cv/raw/water'
WATER_LEAK_DIR = ''

# ============================================================
# Helpers
# ============================================================

def list_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return [
        p for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ]


def find_dirs(root, names):
    root = Path(root)
    if not root.exists():
        return []
    wanted = {n.lower() for n in names}
    return [
        p for p in root.rglob('*')
        if p.is_dir() and p.name.lower() in wanted
    ]


def unique_paths(paths):
    seen = set()
    out = []
    for p in paths:
        p = Path(p)
        key = str(p.resolve()).lower()
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def add_bucket(bucket, label, paths):
    bucket.setdefault(label, []).extend(unique_paths(paths))


def resolve_image(name, search_root):
    name = str(name).strip()
    p = Path(name)

    if p.is_absolute() and p.exists():
        return p

    for candidate in (
        Path(search_root) / name,
        Path(search_root) / 'Images' / name,
        Path(search_root) / 'images' / name,
    ):
        if candidate.exists():
            return candidate

    stem = Path(name).stem
    for candidate in Path(search_root).rglob(f'{stem}.*'):
        if candidate.suffix.lower() in IMG_EXTS:
            return candidate

    return None


def pick_image_col(df):
    for key in ['image', 'file', 'path', 'name']:
        for c in df.columns:
            if key in c.lower():
                return c

    for c in df.columns:
        vals = df[c].astype(str).head(40).tolist()
        if any(Path(v).suffix.lower() in IMG_EXTS for v in vals):
            return c

    return df.columns[0]


def pick_label_col(df, img_col):
    for c in df.columns:
        if c == img_col:
            continue
        if any(k in c.lower() for k in ['label', 'class', 'target', 'category', 'type']):
            return c

    for c in df.columns:
        if c == img_col:
            continue
        vals = df[c].astype(str).str.lower().head(100).tolist()
        if any(('clean' in v) or ('dirty' in v) or ('litter' in v) for v in vals):
            return c

    for c in df.columns:
        if c == img_col:
            continue
        if df[c].nunique(dropna=True) <= 10:
            return c

    return df.columns[-1]


# ============================================================
# Start fresh
# ============================================================

if (DATASET / 'issue').exists():
    shutil.rmtree(DATASET / 'issue')

bucket = {}

# ============================================================
# 1. Pothole / plain road
# ============================================================

pothole_root = RAW / 'pothole'
pothole_imgs = []
plain_imgs = []

for d in find_dirs(pothole_root, ['Pothole', 'Potholes']):
    pothole_imgs += list_images(d)

for d in find_dirs(pothole_root, ['Plain', 'No_Pothole', 'Normal']):
    plain_imgs += list_images(d)

# Fallback: infer from path text
if not pothole_imgs:
    for p in list_images(pothole_root):
        s = str(p).lower()
        if 'pothole' in s:
            pothole_imgs.append(p)

if not plain_imgs:
    for p in list_images(pothole_root):
        s = str(p).lower()
        if 'plain' in s or 'normal' in s:
            plain_imgs.append(p)

add_bucket(bucket, 'pothole', pothole_imgs)
add_bucket(bucket, 'other', plain_imgs)

print('Pothole images:', len(unique_paths(pothole_imgs)))
print('Plain-road images:', len(unique_paths(plain_imgs)))

# ============================================================
# 2. Clean / littered road
# ============================================================

litter_root = RAW / 'litter'
litter_garbage = []
litter_clean = []

meta_files = list(litter_root.rglob('metadata.csv')) if litter_root.exists() else []

if meta_files:
    meta_path = meta_files[0]
    df = pd.read_csv(meta_path)

    print('\nLitter metadata:', meta_path)
    print('Columns:', list(df.columns))
    print(df.head())

    img_col = pick_image_col(df)
    label_col = pick_label_col(df, img_col)

    print('Using image column:', img_col)
    print('Using label column:', label_col)
    print('Label counts:\n', df[label_col].value_counts(dropna=False))

    def map_litter_label(value):
        s = str(value).strip().lower()
        if s in {'0', '0.0'}:
            return 'other'
        if s in {'1', '1.0'}:
            return 'garbage'
        if 'clean' in s:
            return 'other'
        if any(k in s for k in ['dirty', 'litter', 'trash', 'garbage', 'waste']):
            return 'garbage'
        return None

    for _, row in df.iterrows():
        img_path = resolve_image(row[img_col], litter_root)
        label = map_litter_label(row[label_col])
        if img_path is None or label is None:
            continue
        if label == 'garbage':
            litter_garbage.append(img_path)
        else:
            litter_clean.append(img_path)
else:
    # Fallback: infer from path names
    for p in list_images(litter_root):
        s = str(p).lower()
        if any(k in s for k in ['dirty', 'litter', 'trash', 'garbage']):
            litter_garbage.append(p)
        elif 'clean' in s:
            litter_clean.append(p)

add_bucket(bucket, 'garbage', litter_garbage)
add_bucket(bucket, 'other', litter_clean)

print('Litter images:', len(unique_paths(litter_garbage)))
print('Clean-road images:', len(unique_paths(litter_clean)))

# ============================================================
# 3. Extra garbage images from Garbage V2
# ============================================================

if USE_GARBAGE_V2 and (RAW / 'garbage').exists():
    gd_dirs = find_dirs(RAW / 'garbage', ['garbage-dataset'])
    gd_root = gd_dirs[0] if gd_dirs else (RAW / 'garbage')

    class_dirs = [d for d in gd_root.iterdir() if d.is_dir()]

    if class_dirs:
        per_class = max(40, 1200 // len(class_dirs))
        extra = []
        for d in class_dirs:
            imgs = list_images(d)
            random.shuffle(imgs)
            extra.extend(imgs[:per_class])
        add_bucket(bucket, 'garbage', extra)
        print('Extra garbage-v2 images:', len(unique_paths(extra)))

# ============================================================
# 4. Optional water leak
# ============================================================

if WATER_LEAK_DIR and Path(WATER_LEAK_DIR).exists():
    leak_dirs = find_dirs(WATER_LEAK_DIR, ['leak', 'leakage', 'water_leak'])
    water_imgs = []

    if leak_dirs:
        for d in leak_dirs:
            water_imgs += list_images(d)
    else:
        water_imgs = list_images(WATER_LEAK_DIR)

    add_bucket(bucket, 'water_leak', water_imgs)
    print('Water-leak images:', len(unique_paths(water_imgs)))
else:
    print('Water-leak class skipped. Set WATER_LEAK_DIR to enable it.')

# ============================================================
# 5. Filter and split
# ============================================================

bucket = {k: unique_paths(v) for k, v in bucket.items()}
bucket = {k: v for k, v in bucket.items() if len(v) >= MIN_IMAGES_PER_CLASS}

print('\nClasses to train:', sorted(bucket))
for k in sorted(bucket):
    print(f'  {k}: {len(bucket[k])} images')

if len(bucket) < 2:
    raise RuntimeError('Need at least two issue classes. Check dataset download and folder names.')

for split in ['train', 'val', 'test']:
    for label in bucket:
        (DATASET / 'issue' / split / label).mkdir(parents=True, exist_ok=True)

for label, paths in bucket.items():
    random.shuffle(paths)
    paths = paths[:MAX_PER_CLASS]

    n = len(paths)
    n_val = max(1, int(n * 0.15))
    n_test = max(1, int(n * 0.15))
    n_train = n - n_val - n_test

    if n_train < 1:
        n_train = max(1, int(n * 0.7))
        n_val = max(1, int((n - n_train) / 2))
        n_test = n - n_train - n_val

    splits = {
        'train': paths[:n_train],
        'val': paths[n_train:n_train + n_val],
        'test': paths[n_train + n_val:],
    }

    for split_name, split_paths in splits.items():
        for i, src in enumerate(split_paths):
            dest = (
                DATASET / 'issue' / split_name / label
                / f'{label}_{i:06d}{src.suffix.lower()}'
            )
            shutil.copy2(src, dest)

print('\nIssue dataset counts:')
for split in ['train', 'val', 'test']:
    for label in sorted(bucket):
        count = len(list((DATASET / 'issue' / split / label).glob('*')))
        print(f'  {split}/{label}: {count}')


## Step 8 — Visually check the issue dataset

This is important. If the labels are wrong, the model will learn the wrong thing.

Check that:

- `pothole` images show potholes
- `garbage` images show litter or garbage
- `other` images show clean roads or normal scenes
- `water_leak` images show visible water leakage (if enabled)

In [ ]:
def show_samples(split='train', samples_per_class=4):
    split_dir = DATASET / 'issue' / split
    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])

    fig, axes = plt.subplots(
        len(classes),
        samples_per_class,
        figsize=(3 * samples_per_class, 3 * len(classes)),
        squeeze=False,
    )

    for row, cls in enumerate(classes):
        files_for_class = list((split_dir / cls).glob('*'))[:samples_per_class]
        for col in range(samples_per_class):
            ax = axes[row, col]
            ax.axis('off')
            if col < len(files_for_class):
                ax.imshow(Image.open(files_for_class[col]).convert('RGB'))
                ax.set_title(cls)

    plt.tight_layout()
    plt.show()


show_samples()


## Step 9 — Build the AI-image dataset

CIFAKE has this structure:

```text
train/REAL
train/FAKE
test/REAL
test/FAKE
```

This cell converts it into:

```text
ai/train/real
ai/train/fake
ai/val/real
ai/val/fake
ai/test/real
ai/test/fake
```

In [ ]:
if (DATASET / 'ai').exists():
    shutil.rmtree(DATASET / 'ai')

cifake_root = RAW / 'cifake'


def find_one_dir(root, name):
    if root is None:
        return None
    matches = [
        p for p in Path(root).rglob('*')
        if p.is_dir() and p.name.lower() == name.lower()
    ]
    return matches[0] if matches else None


train_root = find_one_dir(cifake_root, 'train')
test_root = find_one_dir(cifake_root, 'test')

print('CIFAKE train root:', train_root)
print('CIFAKE test root:', test_root)

if train_root is None or test_root is None:
    raise RuntimeError(
        'Could not find CIFAKE train/test folders. '
        'Check /content/grievance_cv/raw/cifake.'
    )

real_train_dir = find_one_dir(train_root, 'REAL')
fake_train_dir = find_one_dir(train_root, 'FAKE')
real_test_dir = find_one_dir(test_root, 'REAL')
fake_test_dir = find_one_dir(test_root, 'FAKE')

assert real_train_dir and fake_train_dir and real_test_dir and fake_test_dir, \
    'Missing REAL/FAKE folder'

AI_TRAIN_PER_CLASS = 10000
AI_VAL_PER_CLASS = 2000
AI_TEST_PER_CLASS = 2000


def copy_limited(files, dest_dir, max_n):
    dest_dir.mkdir(parents=True, exist_ok=True)
    random.shuffle(files)
    for i, src in enumerate(files[:max_n]):
        shutil.copy2(src, dest_dir / f'{i:06d}{src.suffix.lower()}')


for split in ['train', 'val', 'test']:
    (DATASET / 'ai' / split / 'real').mkdir(parents=True, exist_ok=True)
    (DATASET / 'ai' / split / 'fake').mkdir(parents=True, exist_ok=True)

real_train = list_images(real_train_dir)
fake_train = list_images(fake_train_dir)
real_test = list_images(real_test_dir)
fake_test = list_images(fake_test_dir)

random.shuffle(real_test)
random.shuffle(fake_test)

copy_limited(
    real_train,
    DATASET / 'ai' / 'train' / 'real',
    AI_TRAIN_PER_CLASS,
)
copy_limited(
    fake_train,
    DATASET / 'ai' / 'train' / 'fake',
    AI_TRAIN_PER_CLASS,
)

val_real = real_test[:AI_VAL_PER_CLASS]
val_fake = fake_test[:AI_VAL_PER_CLASS]

test_real = real_test[
    AI_VAL_PER_CLASS:AI_VAL_PER_CLASS + AI_TEST_PER_CLASS
]
test_fake = fake_test[
    AI_VAL_PER_CLASS:AI_VAL_PER_CLASS + AI_TEST_PER_CLASS
]

copy_limited(val_real, DATASET / 'ai' / 'val' / 'real', AI_VAL_PER_CLASS)
copy_limited(val_fake, DATASET / 'ai' / 'val' / 'fake', AI_VAL_PER_CLASS)
copy_limited(test_real, DATASET / 'ai' / 'test' / 'real', AI_TEST_PER_CLASS)
copy_limited(test_fake, DATASET / 'ai' / 'test' / 'fake', AI_TEST_PER_CLASS)

print('\nAI dataset counts:')
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        count = len(list((DATASET / 'ai' / split / label).glob('*')))
        print(f'  {split}/{label}: {count}')


## Step 10 — Define the model and training function

This is a small CNN trained from random initialization. No pretrained weights are used.

The function:

- Trains with augmentation
- Uses class weights for imbalance
- Tracks validation accuracy
- Applies early stopping
- Evaluates on the held-out test set
- Exports to ONNX
- Saves labels and preprocessing config

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True


class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return (
        total_loss / max(total, 1),
        correct / max(total, 1),
        all_preds,
        all_labels,
    )


def train_model(
    data_dir,
    img_size,
    epochs,
    batch_size,
    output_prefix,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
    lr=1e-3,
    patience=5,
    crop_scale=(0.7, 1.0),
    rotation=10,
):
    data_dir = Path(data_dir)
    train_dir = data_dir / 'train'
    val_dir = data_dir / 'val'
    test_dir = data_dir / 'test'

    if not train_dir.exists():
        raise FileNotFoundError(f'Missing {train_dir}')

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=crop_scale),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(rotation),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = datasets.ImageFolder(train_dir, train_tf)

    val_ok = val_dir.exists() and len(list_images(val_dir)) > 0
    test_ok = test_dir.exists() and len(list_images(test_dir)) > 0

    val_ds = datasets.ImageFolder(val_dir, eval_tf) if val_ok else None
    test_ds = datasets.ImageFolder(test_dir, eval_tf) if test_ok else None

    print('\n' + '=' * 70)
    print('Training:', output_prefix)
    print('Classes:', train_ds.classes)
    print('Train images:', len(train_ds))

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    ) if val_ds else None

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    ) if test_ds else None

    # Class weights for imbalanced classes
    targets = [sample[1] for sample in train_ds.samples]
    counts = np.bincount(
        targets,
        minlength=len(train_ds.classes),
    ).astype(np.float32)

    weights = 1.0 / np.maximum(counts, 1.0)
    weights = weights / weights.sum() * len(weights)

    print('Class counts:', dict(zip(train_ds.classes, counts.astype(int))))
    print('Class weights:', dict(zip(train_ds.classes, np.round(weights, 3))))

    model = SmallCNN(len(train_ds.classes)).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2,
    )

    best_score = -1.0
    best_state = None
    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            train_total += labels.size(0)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)

        if val_loader:
            val_loss, val_acc, _, _ = evaluate(
                model,
                val_loader,
                criterion,
            )
            scheduler.step(val_acc)
            score = val_acc
        else:
            val_loss = float('nan')
            val_acc = float('nan')
            score = train_acc

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
        })

        print(
            f'Epoch {epoch:02d}/{epochs} | '
            f'train_loss={train_loss:.4f} '
            f'train_acc={train_acc:.4f} | '
            f'val_loss={val_loss:.4f} '
            f'val_acc={val_acc:.4f}'
        )

        if score > best_score:
            best_score = score
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            print(f'  -> new best ({best_score:.4f})')

        if epoch - 1 >= patience and val_loader:
            recent = [h['val_acc'] for h in history[-patience:]]
            if max(recent) <= best_score + 1e-4:
                print('Early stopping.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - start_time

    print(
        f'\nTraining finished in {elapsed / 60:.1f} minutes; '
        f'best score={best_score:.4f}'
    )

    # Final evaluation
    eval_loader = test_loader or val_loader or train_loader
    eval_name = 'test' if test_loader else ('val' if val_loader else 'train')

    loss, acc, preds, labels = evaluate(
        model,
        eval_loader,
        criterion,
    )

    print(f'\nFinal {eval_name} accuracy: {acc:.4f}')
    print(classification_report(
        labels,
        preds,
        target_names=train_ds.classes,
        zero_division=0,
    ))
    print('Confusion matrix:')
    print(confusion_matrix(labels, preds))

    # Export to ONNX
    OUTPUT.mkdir(parents=True, exist_ok=True)

    model_cpu = model.to('cpu')
    model_cpu.eval()

    dummy = torch.randn(1, 3, img_size, img_size)

    onnx_path = OUTPUT / f'{output_prefix}.onnx'
    checkpoint_path = OUTPUT / f'{output_prefix}.pt'

    torch.save({
        'state_dict': model_cpu.state_dict(),
        'classes': train_ds.classes,
        'img_size': img_size,
        'mean': list(mean),
        'std': list(std),
    }, checkpoint_path)

    export_kwargs = dict(
        input_names=['input'],
        output_names=['logits'],
        opset_version=17,
        dynamic_axes={
            'input': {0: 'batch'},
            'logits': {0: 'batch'},
        },
    )

    try:
        # Use the legacy exporter to avoid requiring onnxscript on
        # PyTorch versions where the dynamo exporter is the default.
        torch.onnx.export(
            model_cpu,
            dummy,
            str(onnx_path),
            dynamo=False,
            **export_kwargs,
        )
    except TypeError:
        # Older PyTorch versions do not accept the dynamo argument.
        torch.onnx.export(
            model_cpu,
            dummy,
            str(onnx_path),
            **export_kwargs,
        )

    labels_path = OUTPUT / f'{output_prefix}_labels.json'
    preprocess_path = OUTPUT / f'{output_prefix}_preprocess.json'

    with open(labels_path, 'w') as f:
        json.dump(train_ds.classes, f, indent=2)

    with open(preprocess_path, 'w') as f:
        json.dump({
            'img_size': img_size,
            'mean': list(mean),
            'std': list(std),
            'classes': train_ds.classes,
            'input_name': 'input',
            'output_name': 'logits',
        }, f, indent=2)

    # Verify ONNX
    import onnxruntime as ort

    sess = ort.InferenceSession(
        str(onnx_path),
        providers=['CPUExecutionProvider'],
    )

    test_input = np.random.rand(
        1, 3, img_size, img_size,
    ).astype(np.float32)

    test_output = sess.run(
        None,
        {'input': test_input},
    )[0]

    print('ONNX output shape:', test_output.shape)

    history_path = OUTPUT / f'{output_prefix}_history.json'
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)

    print('\nSaved:')
    print(' ', onnx_path)
    print(' ', checkpoint_path)
    print(' ', labels_path)
    print(' ', preprocess_path)
    print(' ', history_path)

    return model, train_ds.classes


## Step 11 — Train the issue classifier

Expected time on a T4: 5–20 minutes.

In [ ]:
issue_model, issue_classes = train_model(
    data_dir=DATASET / 'issue',
    img_size=128,
    epochs=15,
    batch_size=32,
    output_prefix='issue_classifier',
    crop_scale=(0.65, 1.0),
    rotation=10,
)


## Step 12 — Train the AI-image detector

CIFAKE images are `32x32`, so the input size is `32`.

Expected time on a T4: 5–15 minutes.

In [ ]:
ai_model, ai_classes = train_model(
    data_dir=DATASET / 'ai',
    img_size=32,
    epochs=8,
    batch_size=128,
    output_prefix='ai_image_detector',
    mean=(0.5, 0.5, 0.5),
    std=(0.5, 0.5, 0.5),
    crop_scale=(0.8, 1.0),
    rotation=5,
)


## Step 13 — Test both models on uploaded images

Test with:

- A real pothole photo
- A real garbage/litter photo
- A real water-leak photo if you trained that class
- A normal road photo
- An AI-generated image
- A real phone photo

In [ ]:
import onnxruntime as ort
from PIL import Image


def predict(image_path, onnx_path, preprocess_path):
    with open(preprocess_path) as f:
        cfg = json.load(f)

    img_size = cfg['img_size']
    mean = np.array(cfg['mean'], dtype=np.float32)
    std = np.array(cfg['std'], dtype=np.float32)
    classes = cfg['classes']

    img = Image.open(image_path).convert('RGB')

    try:
        resample = Image.Resampling.BILINEAR
    except AttributeError:
        resample = Image.BILINEAR

    img = img.resize((img_size, img_size), resample)

    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = (arr - mean) / std
    arr = np.transpose(arr, (2, 0, 1))[None, ...].astype(np.float32)

    sess = ort.InferenceSession(
        str(onnx_path),
        providers=['CPUExecutionProvider'],
    )

    logits = sess.run(
        None,
        {cfg['input_name']: arr},
    )[0][0]

    probs = np.exp(logits - logits.max())
    probs = probs / probs.sum()

    order = np.argsort(-probs)

    for idx in order:
        print(f'  {classes[idx]}: {probs[idx]:.4f}')

    return classes[int(order[0])], float(probs[order[0]])


from google.colab import files
uploaded_test = files.upload()

for name in uploaded_test:
    print('\n' + '=' * 70)
    print(name)

    print('\nIssue prediction:')
    issue_label, issue_conf = predict(
        name,
        OUTPUT / 'issue_classifier.onnx',
        OUTPUT / 'issue_classifier_preprocess.json',
    )

    print('\nAI-image prediction:')
    ai_label, ai_conf = predict(
        name,
        OUTPUT / 'ai_image_detector.onnx',
        OUTPUT / 'ai_image_detector_preprocess.json',
    )

    print(
        f'\nResult: issue={issue_label} '
        f'({issue_conf:.2%}), AI={ai_label} ({ai_conf:.2%})'
    )


## Step 14 — Save outputs to Google Drive and download a zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import zipfile

drive_out = Path('/content/drive/MyDrive/grievance_cv_outputs')

if drive_out.exists():
    shutil.rmtree(drive_out)

shutil.copytree(OUTPUT, drive_out)
print('Copied to:', drive_out)

zip_path = Path('/content/grievance_cv_outputs.zip')

with zipfile.ZipFile(
    zip_path,
    'w',
    zipfile.ZIP_DEFLATED,
) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(OUTPUT))

print('Zip created:', zip_path)

from google.colab import files
files.download(str(zip_path))


## What to send for integration

Send these files:

```text
issue_classifier.onnx
issue_classifier_labels.json
issue_classifier_preprocess.json

ai_image_detector.onnx
ai_image_detector_labels.json
ai_image_detector_preprocess.json
```

Also send:

- Final test accuracy for both models
- Classification report or confusion matrix
- 2–3 test images per issue class
- 2–3 real and 2–3 AI-generated test images

## Troubleshooting

### `403 Forbidden` from Kaggle

The `kaggle.json` token was not accepted. Re-upload it and rerun Step 3.

### `FileNotFoundError` or `Found 0 files`

Run the inspection cell in Step 6. Dataset folder names sometimes differ.

### `CUDA out of memory`

Reduce the issue-model batch size:

```python
batch_size=16
```

### Clean and dirty labels are reversed

In Step 7, swap the `0` and `1` mapping inside `map_litter_label`, then rerun Step 7.

### Water-leak accuracy is poor

Expected with a small Roboflow dataset. Either collect more leak images or leave `water_leak` out of the first version.

## Important limitations

- The issue classifier is trained from scratch on a small dataset. It is demo-grade.
- CIFAKE is `32x32`. The AI detector may not generalise to real phone photos.
- Never auto-reject a complaint based on the AI-generated score.
- Use the image result as supporting evidence, not proof.
